# Imports and Constants

In [4]:
# The openactive package
import openactive as oa
# To harvest the current date
from datetime import date
# Requests and retry for RPDE feeds
import requests
from urllib3.util import Retry
# Import json for formatting
import json
import pandas as pd
# import beautiful soup for web scraping links
from bs4 import BeautifulSoup

# Get current date to track when the dataset is downloaded
HARVEST_DATE = date.today().isoformat()

# A function for printing out formatted output as suggested by the OA package github
def printer(arg):
    print(json.dumps(arg, indent=4))

## Get the table of all providers and feeds off of OpenActive's website

In [5]:
url = "https://status.openactive.io/"

# Get webpage
response = requests.get(url, timeout=30)
response.raise_for_status()

# Parse HTML
soup = BeautifulSoup(response.text, "html.parser")

# Find tables
tables = soup.find_all("table")

# Load the table into a Pandas DataFrame

In [6]:
# instantiate the data, each row of the table will be appended to this
data = []

# go through the first table
for table in tables[:1]:
    # Skip tables without headers
    headers = [th.get_text(strip=True) for th in table.find_all("th")]
    
    # Get the index for the column which "Provider" and "Feed Status" is in
    provider_index = headers.index("Provider")
    feed_index = headers.index("Feed status")
    
    # Loop through each row of the table
    for row in table.find_all("tr"):
        # For each row, get all columns
        cells = row.find_all("td")
        
        if len(cells) <= max(provider_index, feed_index):
            continue
        
        # Get the <a> marker that includes both href and anchor text
        provider_cell = cells[provider_index].find_all("a")[1]
        # Get the URL from the href
        provider_url = provider_cell.get("href")
        # Get the name through the anchor text 
        provider_name = provider_cell.text
        
        # Instantiate the "Feed status" column as feed_cell
        feed_cell = cells[feed_index]
        
        # Set the following feeds as None
        facilityUse = None
        slot = None
        sessionSeries = None
        scheduledSession = None
        event = None
        unnamedFeed = None
        
        # Loop through each <a> in feed_cell, 
        # if there is a valid link update the feeds above.
        # Otherwise, None will represent lack of feed
        for a in feed_cell.find_all("a"):
            if a.text == "FacilityUse":
                facilityUse = a.get("href")
            elif a.text == "Slot":
                slot = a.get("href")
            elif a.text == "SessionSeries":
                sessionSeries = a.get("href")
            elif a.text == "ScheduledSession":
                scheduledSession = a.get("href")
            elif a.text == "Event":
                event = a.get("href")
            elif a.text == "Unnamed Feed":
                unnamedFeed = a.get("href")
        
        # Append all the data scraped from the above code into the data list 
        data.append({
            "provider": provider_name,
            "provider_url": provider_url,
            "facility_use": facilityUse,
            "slot": slot,
            "sessionSeries": sessionSeries,
            "scheduledSession": scheduledSession,
            "event": event,
            "unnamedFeed": unnamedFeed
        })

# Create DataFrame from the final data list after running the code above
df = pd.DataFrame(data)

## Preliminary inspection of the loaded DF

In [7]:
# display the first few rows of the df
df.head()

,provider,provider_url,facility_use,slot,sessionSeries,scheduledSession,event,unnamedFeed
0,100% TO THE TOP CIC,https://topcic.bookteq.com/api/open-active/,https://topcic.bookteq.com/api/open-active/fac...,https://topcic.bookteq.com/api/open-active/slots,NaN,NaN,NaN,NaN
1,Actihire,https://actihire.bookteq.com/api/open-active/,https://actihire.bookteq.com/api/open-active/f...,https://actihire.bookteq.com/api/open-active/s...,NaN,NaN,NaN,NaN
2,Active Hartlepool,https://activehartlepool.gs-signature.cloud/Op...,https://opendata.leisurecloud.live/api/feeds/H...,https://opendata.leisurecloud.live/api/feeds/H...,https://opendata.leisurecloud.live/api/feeds/H...,https://opendata.leisurecloud.live/api/feeds/H...,NaN,NaN
3,Active Leeds,https://activeleeds-oa.leisurecloud.net/OpenAc...,https://opendata.leisurecloud.live/api/feeds/A...,https://opendata.leisurecloud.live/api/feeds/A...,https://opendata.leisurecloud.live/api/feeds/A...,https://opendata.leisurecloud.live/api/feeds/A...,NaN,NaN
4,Active Luton,https://activeluton-openactive.legendonlineser...,https://activeluton-openactive.legendonlineser...,https://activeluton-openactive.legendonlineser...,https://activeluton-openactive.legendonlineser...,NaN,NaN,NaN


In [8]:
df.describe()

,provider,provider_url,facility_use,slot,sessionSeries,scheduledSession,event,unnamedFeed
count,174,174,149,148,72,38,8,9
unique,171,174,148,147,71,37,8,9
top,Chelmsford City Sports,https://topcic.bookteq.com/api/open-active/,https://opendata.leisurecloud.live/api/feeds/C...,https://opendata.leisurecloud.live/api/feeds/C...,https://opendata.leisurecloud.live/api/feeds/C...,https://opendata.leisurecloud.live/api/feeds/C...,https://bookwhen.com/api/openactive/events,http://api.letsride.co.uk/public/v1/rides
freq,2,1,2,2,2,2,1,1
